# Kaggle histopathology introduction
This notebook is an introduction to the data challenge of out of distribution classification of histopathology patches. It also serves as a baseline for the code and the model.

If you have any questions, feel free to contact me at [leo.fillioux@centralesupelec.fr](mailto:leo.fillioux@centralesupelec.fr).

In [ ]:
import h5py
import torch
import random
import numpy as np
import pandas as pd
import torchmetrics
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

In [ ]:
TRAIN_IMAGES_PATH = 'train.h5'
VAL_IMAGES_PATH = 'val.h5'
TEST_IMAGES_PATH = 'test.h5'
SEED = 0

In [ ]:
torch.random.manual_seed(SEED)
random.seed(SEED)

## 1. Introduction to the data
The dataset consists of patches of whole slide images which should be classified into either containing tumor or not. The training images come from 3 different centers (i.e. hospitals), while the validation set comes from another center and the test set from yet another center. The visual aspect of the patches are quite different due to the slightly different staining procedures, conditions, and equipment from each hospital. The objective of the task is to build a classifier that is impacted by this distribution shift as little as possible.

The data is stored in `.h5` files, which can be seen as a folder hierarchy, which are can be seen as the following.
```
├── idx           # index of the image
│   └── img       # image in a tensor format
│   └── label     # binary label of the image
│   └── metadata  # some metadata on the images
```
The metadata is included for completeness but is not necessarily useful. The first element in the metadata corresponds to the center.

The following is a visualization of how different the images look from the different centers.

In [ ]:
train_images = {0: {0: None, 1: None},
                3: {0: None, 1: None},
                4: {0: None, 1: None}}
val_images = {1: {0: None, 1: None}}

In [ ]:
for img_data, data_path in zip([train_images, val_images], [TRAIN_IMAGES_PATH, VAL_IMAGES_PATH]):
    with h5py.File(data_path, 'r') as hdf:
        for img_idx in list(hdf.keys()):
            label = int(np.array(hdf.get(img_idx).get('label')))
            center = int(np.array(hdf.get(img_idx).get('metadata'))[0])
            if img_data[center][label] is None:
                img_data[center][label] = np.array(hdf.get(img_idx).get('img'))
            if all(all(value is not None for value in inner_dict.values()) for inner_dict in img_data.values()):
                break
all_data = {**train_images, **val_images}

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(20, 10))
center_ids = {center: idx for idx, center in enumerate(all_data.keys())}
for center in all_data:
    for label in all_data[center]:
        axs[label, center_ids[center]].imshow(np.moveaxis(all_data[center][label], 0, -1).astype(np.float32))
        axs[label, center_ids[center]].axis('off')
        if label == 0:
            axs[label, center_ids[center]].set_title(f'Center {center}')
plt.show()

## 2. Imports and Hyperparameters
All solutions (including datasets, models, training, evaluation) now live in `src/`, but the notebook stays the main place to run it. Therefore, we must import all scripts from `src/` folder.

In [ ]:
import importlib
import src.lib.augmentations as augmentations_module
import src.lib.datasets as datasets_module
import src.lib.solutions as solutions_module
import src.main as main_module

importlib.reload(augmentations_module)
importlib.reload(datasets_module)
importlib.reload(solutions_module)
importlib.reload(main_module)

from src.main import get_solution

Here, we define our hyperparameters used throughout the notebook.

In [ ]:
BATCH_SIZE = 16
CORAL_BATCH_SIZE = 18
HEAD_LR = 1e-3
BACKBONE_LR = 4e-4
LORA_RANK = 4
LORA_ALPHA = 1.0

## 3. Baseline solution
This solution consists of a frozen DINOv2 ViT-S/14 backbone with a linear classification head. Images are used at 98x98 resolution and normalized with ImageNet statistics, with no other augmentations applied.

In [ ]:
CONFIG = {
    'train_path': TRAIN_IMAGES_PATH,
    'val_path': VAL_IMAGES_PATH,
    'test_path': TEST_IMAGES_PATH,
    'output_csv': 'baseline.csv',
    'batch_size': BATCH_SIZE,
    'resize': (98, 98),
    'num_epochs': 100,
    'patience': 10,
    'lr': 0.001,
}

solution = get_solution('baseline', CONFIG)

#### Training

In [ ]:
history = solution.fit()

#### Making the final prediction

In [ ]:
solution.predict_test(output_csv='baseline.csv')

## 4. Baseline 224 Solution
This solution is identical to the baseline, but uses higher input resolution of 224x224.

In [ ]:
HIGHRES_CONFIG = {
    "train_path": TRAIN_IMAGES_PATH,
    "val_path": VAL_IMAGES_PATH,
    "test_path": TEST_IMAGES_PATH,
    "output_csv": "baseline_224.csv",
    "batch_size": BATCH_SIZE,
    "num_epochs": 100,
    "patience": 10,
    "lr": 0.001,
    "checkpoint_path": "best_model_224.pth",
}

highres_solution = get_solution("baseline_224", HIGHRES_CONFIG)

#### Training

In [ ]:
highres_solution.fit()

#### Making the final prediction

In [ ]:
highres_solution.predict_test(output_csv="baseline_224.csv")

## 5. Baseline + Color Jitter Solution
This is baseline solution, with added train-only color jitter augmentation (i.e., brightness, contrast, saturation, hue). The backbone is still frozen, but features are recomputed every epoch. 

In [ ]:
COLOR_JITTER_CONFIG = {
    "train_path": TRAIN_IMAGES_PATH,
    "val_path": VAL_IMAGES_PATH,
    "test_path": TEST_IMAGES_PATH,
    "output_csv": "baseline_color_jitter.csv",
    "batch_size": BATCH_SIZE,
    "resize": (98, 98),
    "num_epochs": 100,
    "patience": 10,
    "lr": 0.001,
    "checkpoint_path": "best_model_color_jitter.pth",
    "jitter_brightness": 0.15,
    "jitter_contrast": 0.15,
    "jitter_saturation": 0.15,
    "jitter_hue": 0.02,
}

color_jitter_solution = get_solution("baseline_color_jitter", COLOR_JITTER_CONFIG)

#### Training

In [ ]:
color_jitter_solution.fit()

#### Making the final prediction

In [ ]:
color_jitter_solution.predict_test(output_csv="baseline_color_jitter.csv")

## 6. Baseline 224 + Targeted Augmentations
This solution extends the previous _Baseline 224_ with targeted augmentations designed for histopathology task. Specifically, this solution introduces Stain Jitter augmentation, together with random flips, rotations, and previously used Color Jitter. It also supports the use of Reinhard normalization (in form of augmentation), which is enabled through flag `use_reinhard=True`.

The trained model can be used both for making the final prediction, but is primarily used in the first stage of LP-FT framework.

In [ ]:
TARGETED_224_CONFIG = {
    "train_path": TRAIN_IMAGES_PATH,
    "val_path": VAL_IMAGES_PATH,
    "test_path": TEST_IMAGES_PATH,
    "output_csv": "baseline_224_targeted_augmentations.csv",
    "batch_size": BATCH_SIZE,
    "resize": (224, 224),
    "num_epochs": 100,
    "patience": 10,
    "head_lr": 0.001,
    "checkpoint_path": "best_model_224_targeted_augmentations.pth",
    "stain_sigma": 0.1,
    "jitter_brightness": 0.15,
    "jitter_contrast": 0.15,
    "use_reinhard": False
}

targeted_224_solution = get_solution("baseline_224_targeted_augmentations", TARGETED_224_CONFIG)

#### Training

In [ ]:
targeted_224_history = targeted_224_solution.fit()

#### Making the final prediction

In [ ]:
targeted_224_solution.predict_test(output_csv="baseline_224_targeted_augmentations.csv")

## 7. LoRA DINOv2 + Targeted Augmentations

In this solution, rather than keeping the backbone frozen, we fine-tune DINOv2 using LoRA. LoRA layers are injected into the query, key, value, and projection layers of each attention block, while the rest of the backbone remains frozen. Moreover, the same targeted augmentation pipeline from the previous solution is used. This solution also enables Reinhard normalization via `use_reinhard: True`, and a cosine annealing scheduler, which can  be enabled through `scheduler: "cosine"`.

This solution is intended as the second stage of the LP-FT framework. Therefore, it loads the classifier head weights from the targeted-augmentation linear probe checkpoint (via `checkpoint_head` parameter) and performs end-to-end training. This is done in order to warm-up the classifier head, so it is not randomly initialized.


In [ ]:
LORA_TARGETED_CONFIG = {
    "train_path": TRAIN_IMAGES_PATH,
    "val_path": VAL_IMAGES_PATH,
    "test_path": TEST_IMAGES_PATH,
    "batch_size": BATCH_SIZE,
    "num_epochs": 100,
    "patience": 10,
    "head_lr": 0.001,
    "backbone_lr": 4e-4,
    "checkpoint_path": "best_model_lora_dinov2_targeted_augmentations.pth",
    "checkpoint_head": "best_model_224_targeted_augmentations.pth",
    "lora_rank": 4,
    "lora_alpha": 1.0,
    "stain_sigma": 0.1,
    "jitter_brightness": 0.15,
    "jitter_contrast": 0.15,
    "scheduler": "cosine",
    "use_reinhard": False,
}

lora_targeted_solution = get_solution("lora_dinov2_targeted_augmentations", LORA_TARGETED_CONFIG)

#### Training

In [ ]:
lora_targeted_history = lora_targeted_solution.fit()

#### Making the final prediction

In [ ]:
lora_targeted_solution.predict_test(output_csv="lora_dinov2_targeted_augmentations.csv")

## 8. LoRA DINOv2 + Class-conditional CORAL (concise run)

This solution extends the last one by incorporating class-conditional CORAL regularization across training centers in to the end-to-end training. Here, the batch size must be divisible by the number of train centers for balanced center sampling.

In [ ]:
LORA_CC_CORAL_CONFIG = {
    "train_path": TRAIN_IMAGES_PATH,
    "val_path": VAL_IMAGES_PATH,
    "test_path": TEST_IMAGES_PATH,
    "output_csv": "lora_dinov2_class_conditional_coral.csv",
    "checkpoint_path": "best_model_lora_dinov2_class_conditional_coral.pth",
    "head_init_checkpoint": "best_model_224_targeted_augmentations.pth",
    "batch_size": CORAL_BATCH_SIZE,
    "num_epochs": 100,
    "patience": 10,
    "head_lr": 1e-3,
    "backbone_lr": 4e-4,
    "lora_rank": 8,
    "lora_alpha": 1.0,
    "coral_lambda": 0.05,
    "coral_eps": 1e-5,
    "num_workers": 4,
}

lora_cc_coral_solution = get_solution("lora_dinov2_class_conditional_coral", LORA_CC_CORAL_CONFIG)


#### Training

In [ ]:
lora_cc_coral_history = lora_cc_coral_solution.fit()

#### Making the final prediction

In [ ]:
lora_cc_coral_solution.predict_test(output_csv="lora_dinov2_class_conditional_coral.csv")

## 9. UNI + Targeted Augmentations
This solution reuses the stage-1 targeted-augmentation checkpoint to warm-start the classifier head, then fine-tunes a UNI backbone with LoRA adapters. It keeps the same histopathology preprocessing setup, including optional Reinhard normalization through the solution config.

In [ ]:
UNI_CONFIG = {
    'train_path': TRAIN_IMAGES_PATH,
    'val_path': VAL_IMAGES_PATH,
    'test_path': TEST_IMAGES_PATH,
    'output_csv': 'uni_targeted_augmentations.csv',
    'batch_size': BATCH_SIZE,
    'num_epochs': 100,
    'patience': 10,
    'head_lr': 1e-3,
    'backbone_lr': 1e-4,
    'lora_rank': 8,
    'lora_alpha': 1.0,
    'backbone_name': 'uni',
    'backbone_kwargs': {
        'model_name': 'hf-hub:MahmoodLab/UNI',
        'init_values': 1e-5,
        'dynamic_img_size': True,
    },
    'use_reinhard': True,
    'optimizer_name': 'adamw',
    'weight_decay': 1e-4,
    'scheduler': 'cosine',
    'warmup_epochs': 5,
    'head_init_checkpoint': 'best_model_224_targeted_augmentations.pth',
    'checkpoint_path': 'best_model_lora_uni_targeted_augmentations.pth',
}

uni_solution = get_solution('lora_uni_targeted_augmentations', UNI_CONFIG)

#### Training

In [ ]:
uni_solution.fit()

#### Making the final prediction

In [ ]:
uni_solution.predict_test(output_csv='uni_targeted_augmentations.csv')